# Rice Type Classification Using PyTorch

## Project Objective
The objective of this project is to build a deep learning model using PyTorch to classify rice grains into their respective rice types based on their morphological features.

## Dataset
The project uses the **Rice Type Classification Dataset**, containing morphological measurements of rice grains. The input features include measurements such as:
- Area
- Major Axis Length
- Minor Axis Length
- Eccentricity
- Convex Area
- EquivDiameter
- Extent
- Perimeter
- Roundness
- Aspect Ratio
- And other extracted grain characteristics

The target variable represents the rice type/class.

## PyTorch Model
A neural network is implemented using **PyTorch**. The workflow includes:
- Custom dataset preparation
- Data loading using PyTorch `DataLoader`
- Feature preprocessing
- Neural network construction
- Forward propagation
- Loss calculation
- Backpropagation and parameter optimization

## Training Approach
The dataset is divided into training and testing sets. The model is trained using mini-batches through PyTorch's `DataLoader`.

During training, the model:
1. Receives batches of rice-grain features.
2. Performs forward propagation.
3. Calculates the classification loss.
4. Computes gradients using backpropagation.
5. Updates model parameters using an optimizer.

## Evaluation Metric
The primary evaluation metric is **classification accuracy**, which measures the percentage of rice samples correctly classified by the trained model.

The model's performance is evaluated on the unseen test dataset to determine its generalization ability.

## Importing Dataset

In [1]:
pip install opendatasets --quiet

In [2]:
import opendatasets as od

In [3]:
od.download('https://www.kaggle.com/datasets/mssmartypants/rice-type-classification')

Skipping, found downloaded files in "./rice-type-classification" (use force=True to force download)


## Importing Libraries

In [4]:
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader
from torchsummary import summary
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


## EDA

In [5]:
df = pd.read_csv('/content/rice-type-classification/riceClassification.csv')

In [6]:
df.head()

,id,Area,MajorAxisLength,MinorAxisLength,Eccentricity,ConvexArea,EquivDiameter,Extent,Perimeter,Roundness,AspectRation,Class
0,1,4537,92.229316,64.012769,0.719916,4677,76.004525,0.657536,273.085,0.764510,1.440796,1
1,2,2872,74.691881,51.400454,0.725553,3015,60.471018,0.713009,208.317,0.831658,1.453137,1
2,3,3048,76.293164,52.043491,0.731211,3132,62.296341,0.759153,210.012,0.868434,1.465950,1
3,4,3073,77.033628,51.928487,0.738639,3157,62.551300,0.783529,210.657,0.870203,1.483456,1
4,5,3693,85.124785,56.374021,0.749282,3802,68.571668,0.769375,230.332,0.874743,1.510000,1


In [7]:
df.tail()

,id,Area,MajorAxisLength,MinorAxisLength,Eccentricity,ConvexArea,EquivDiameter,Extent,Perimeter,Roundness,AspectRation,Class
18180,18181,5853,148.624571,51.029281,0.939210,6008,86.326537,0.498594,332.960,0.663444,2.912535,0
18181,18182,7585,169.593996,58.141659,0.939398,7806,98.272692,0.647461,385.506,0.641362,2.916910,0
18182,18183,6365,154.777085,52.908085,0.939760,6531,90.023162,0.561287,342.253,0.682832,2.925396,0
18183,18184,5960,151.397924,51.474600,0.940427,6189,87.112041,0.492399,343.371,0.635227,2.941216,0
18184,18185,6134,153.081981,51.590606,0.941500,6283,88.374495,0.489975,338.613,0.672274,2.967245,0


In [8]:
df.shape

(18185, 12)

In [9]:
df.isnull().sum()

,0
id,0
Area,0
MajorAxisLength,0
MinorAxisLength,0
Eccentricity,0
ConvexArea,0
EquivDiameter,0
Extent,0
Perimeter,0
Roundness,0


In [10]:
df.columns

Index(['id', 'Area', 'MajorAxisLength', 'MinorAxisLength', 'Eccentricity',
       'ConvexArea', 'EquivDiameter', 'Extent', 'Perimeter', 'Roundness',
       'AspectRation', 'Class'],
      dtype='object')

In [11]:
x = df['Class'].value_counts()

In [12]:
x

,count
Class,
1,9985
0,8200


In [13]:
(x/len(df.Class))*100

,count
Class,
1,54.907891
0,45.092109


In [14]:
df.describe()

,id,Area,MajorAxisLength,MinorAxisLength,Eccentricity,ConvexArea,EquivDiameter,Extent,Perimeter,Roundness,AspectRation,Class
count,18185.000000,18185.000000,18185.000000,18185.000000,18185.000000,18185.000000,18185.000000,18185.000000,18185.000000,18185.000000,18185.000000,18185.000000
mean,9093.000000,7036.492989,151.680754,59.807851,0.915406,7225.817872,94.132952,0.616653,351.606949,0.707998,2.599081,0.549079
std,5249.701658,1467.197150,12.376402,10.061653,0.030575,1502.006571,9.906250,0.104389,29.500620,0.067310,0.434836,0.497599
min,1.000000,2522.000000,74.133114,34.409894,0.676647,2579.000000,56.666658,0.383239,197.015000,0.174590,1.358128,0.000000
25%,4547.000000,5962.000000,145.675910,51.393151,0.891617,6125.000000,87.126656,0.538530,333.990000,0.650962,2.208527,0.000000
50%,9093.000000,6660.000000,153.883750,55.724288,0.923259,6843.000000,92.085696,0.601194,353.088000,0.701941,2.602966,1.000000
75%,13639.000000,8423.000000,160.056214,70.156593,0.941372,8645.000000,103.559146,0.695664,373.003000,0.769280,2.964101,1.000000
max,18185.000000,10210.000000,183.211434,82.550762,0.966774,11008.000000,114.016559,0.886573,508.511000,0.904748,3.911845,1.000000


In [15]:
df.duplicated().sum()

np.int64(0)

In [16]:
new_df = df.drop('id',axis = 1)

In [17]:
new_df.shape

(18185, 11)

In [18]:
from sklearn.preprocessing import StandardScaler
x = new_df.drop('Class',axis = 1)
y = new_df['Class']
st = StandardScaler()
x = st.fit_transform(x)

In [19]:
x.dtype

dtype('float64')

In [20]:
x = np.array(new_df.iloc[:,:-1])
y = np.array(new_df.iloc[:,-1])

In [21]:
features=pd.DataFrame(x)

In [22]:
label = pd.DataFrame(y)

## Train-Test-Split

In [23]:
x_train,x_test,y_train,y_test = train_test_split(x,y,train_size=0.7,random_state=50)

In [24]:
x_test,x_val,y_test,y_val = train_test_split(x_test, y_test, test_size=0.5,random_state=50)

In [25]:
print(x_train.shape)
print(x_test.shape)
print(x_val.shape)

(12729, 10)
(2728, 10)
(2728, 10)


In [26]:
x_train = st.fit_transform(x_train)
x_test = st.transform(x_test)
x_val = st.transform(x_val)

## DataSet & DataLoader

In [27]:
class dtset(Dataset):
  def __init__(self,x,y):
    self.x = torch.tensor(x,dtype = torch.float32).to(device)
    self.y = torch.tensor(y,dtype = torch.float32).to(device)
  def __len__(self):
    return len(self.x)
  def __getitem__(self,idx):
    return self.x[idx],self.y[idx]

In [28]:
training_dataset = dtset(x_train,y_train)
testing_dataset = dtset(x_test,y_test)
validation_dataset = dtset(x_val,y_val)

In [29]:
train_dataloader = DataLoader(training_dataset, batch_size = 32 , shuffle = True)
test_dataloader = DataLoader(testing_dataset, batch_size = 32 , shuffle = True)
val_dataloader = DataLoader(validation_dataset, batch_size = 32 , shuffle = True)

In [30]:
for x,y in train_dataloader:
  print(x)
  print('====')
  print(y)
  break

tensor([[-0.1393,  0.8100, -0.5213,  0.8498, -0.1621, -0.0869, -0.8600,  0.3480,
         -0.7835,  0.8432],
        [-0.5991,  0.3258, -0.8304,  0.9294, -0.5814, -0.5634, -1.1557, -0.1305,
         -1.0168,  0.9856],
        [-0.9538, -0.9549, -0.7502,  0.3944, -0.9504, -0.9492, -0.7571, -1.0351,
         -0.3035,  0.1739],
        [-0.8948,  0.0902, -1.0897,  1.0497, -0.8801, -0.8838, -1.5341, -0.3296,
         -1.3812,  1.2195],
        [-0.5333,  0.1347, -0.6286,  0.7129, -0.5158, -0.4937,  1.1717, -0.1908,
         -0.7665,  0.6183],
        [ 1.2054,  0.8981,  1.0112, -0.4046,  1.2999,  1.1849,  1.6486,  1.5998,
         -0.1930, -0.6245],
        [-1.4807, -0.5984, -1.5830,  1.2146, -1.4949, -1.5573, -0.9266, -1.2032,
         -1.4322,  1.5841],
        [-0.9334, -0.5705, -0.8802,  0.6675, -0.9239, -0.9266,  0.3333, -0.8468,
         -0.5988,  0.5488],
        [-1.0108, -0.8577, -0.8972,  0.5711, -1.0391, -1.0129, -0.5403, -1.1623,
         -0.2139,  0.4083],
        [-1.6584, -

## Model

In [31]:
hidden_neurons = 10
class Model(nn.Module):
  def __init__(self):
    super(Model, self).__init__()

    self.input_layer = nn.Linear(x.shape[1],hidden_neurons)
    self.linear = nn.Linear(hidden_neurons,1)
    self.sigmoid = nn.Sigmoid()

  def forward(self,x):
    x = self.input_layer(x)
    x = self.linear(x)
    x = self.sigmoid(x)
    return x

In [32]:
model = Model().to(device)

In [33]:
summary(model,(x.shape[1],))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1                   [-1, 10]             110
            Linear-2                    [-1, 1]              11
           Sigmoid-3                    [-1, 1]               0
Total params: 121
Trainable params: 121
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00
----------------------------------------------------------------


## Loss & Optimzer

In [34]:
criterion = nn.BCELoss()
optimizer = Adam(model.parameters(), lr = 0.001)

## Training

In [35]:
EPOCHS = 10
for epoch in range(EPOCHS):
    total_acc_train = 0
    total_loss_train = 0
    total_acc_val = 0
    total_loss_val = 0
    for data in train_dataloader:

        inputs, labels = data

        prediction = model(inputs).squeeze(1)

        batch_loss = criterion(prediction, labels)

        total_loss_train += batch_loss.item()

        acc = ((prediction).round() == labels).sum().item()

        total_acc_train += acc

        batch_loss.backward()
        optimizer.step()
        optimizer.zero_grad()

    with torch.no_grad():
        for data in val_dataloader:
            inputs, labels = data

            prediction = model(inputs).squeeze(1)

            batch_loss = criterion(prediction, labels)

            total_loss_val += batch_loss.item()

            acc = ((prediction).round() == labels).sum().item()

            total_acc_val += acc


    print(f'''Epoch no. {epoch + 1} Train Loss: {total_loss_train/1000:.4f} Train Accuracy: {(total_acc_train/(training_dataset.__len__())*100):.4f} Validation Loss: {total_loss_val/1000:.4f} Validation Accuracy: {(total_acc_val/(validation_dataset.__len__())*100):.4f}''')
    print("="*50)


Epoch no. 1 Train Loss: 0.0797 Train Accuracy: 95.0664 Validation Loss: 0.0048 Validation Accuracy: 98.9370
Epoch no. 2 Train Loss: 0.0202 Train Accuracy: 98.4209 Validation Loss: 0.0034 Validation Accuracy: 98.7903
Epoch no. 3 Train Loss: 0.0173 Train Accuracy: 98.5466 Validation Loss: 0.0031 Validation Accuracy: 98.8636
Epoch no. 4 Train Loss: 0.0164 Train Accuracy: 98.6330 Validation Loss: 0.0030 Validation Accuracy: 98.8270
Epoch no. 5 Train Loss: 0.0159 Train Accuracy: 98.6645 Validation Loss: 0.0030 Validation Accuracy: 98.8270
Epoch no. 6 Train Loss: 0.0156 Train Accuracy: 98.7037 Validation Loss: 0.0029 Validation Accuracy: 98.8270
Epoch no. 7 Train Loss: 0.0153 Train Accuracy: 98.7430 Validation Loss: 0.0031 Validation Accuracy: 98.8270
Epoch no. 8 Train Loss: 0.0151 Train Accuracy: 98.7195 Validation Loss: 0.0029 Validation Accuracy: 98.9370
Epoch no. 9 Train Loss: 0.0149 Train Accuracy: 98.7980 Validation Loss: 0.0028 Validation Accuracy: 98.9370
Epoch no. 10 Train Loss: 0.0

## Testing

In [36]:
with torch.no_grad():
  total_loss_test = 0
  total_acc_test = 0
  for data in test_dataloader:
    inputs, labels = data

    prediction = model(inputs).squeeze(1)

    batch_loss_test = criterion((prediction), labels)
    total_loss_test += batch_loss_test.item()
    acc = ((prediction).round() == labels).sum().item()
    total_acc_test += acc

print(f"Accuracy Score is: {round((total_acc_test/x_test.shape[0])*100, 2)}%")

Accuracy Score is: 98.9%


# Conclusion

In this project, a neural network was developed using **PyTorch** to classify rice types based on morphological characteristics of rice grains. The complete deep learning workflow was implemented, including data preprocessing, dataset preparation, batch loading, model training, loss calculation, backpropagation, and evaluation.

This project provided practical experience with the fundamental components of a PyTorch-based machine learning pipeline and demonstrated how a neural network can be trained for a binary classification task using tabular data.

Overall, the project establishes a foundation for applying PyTorch to more advanced deep learning problems, including image classification and computer vision tasks.